In [1]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types
from pyspark.sql.functions import col

In [2]:
pyspark.__file__

'/home/daniel/anaconda3/lib/python3.13/site-packages/pyspark/__init__.py'

In [3]:
spark = SparkSession.builder \
        .master('spark://de-zoomcamp.us-central1-c.c.project-0c3c5223-416f-4242-b0f.internal:7077') \
        .appName('test') \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 02:50:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark

In [5]:
df_green = \
    spark.read \
    .option('header', 'true') \
    .option('recursiveFileLookup', 'true') \
    .parquet('/home/daniel/notebooks/taxi_data_pq/green/')


In [6]:
df_yellow = \
    spark.read \
    .option('header', 'true') \
    .option('recursiveFileLookup', 'true') \
    .parquet('/home/daniel/notebooks/taxi_data_pq/yellow/')

In [7]:
# normalizing green columns: 
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('taxi_type', F.lit('green')) \
    .select(
        'VendorID', 
        'pickup_datetime', 
        'dropoff_datetime', 
        'store_and_fwd_flag',
        'RatecodeID',
        'PULocationID',
        'DOLocationID',
        'passenger_count',
        'trip_distance',
        'fare_amount',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'ehail_fee',
        'improvement_surcharge',
        'total_amount',
        'payment_type',
        'trip_type',
        'congestion_surcharge',
        'taxi_type'
    )

In [8]:
# normalizing yellow columns: 
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('taxi_type', F.lit('yellow')) \
    .select(
        'VendorID', 
        'pickup_datetime', 
        'dropoff_datetime', 
        'passenger_count',
        'trip_distance',
        'RatecodeID',
        'store_and_fwd_flag',
        'PULocationID',
        'DOLocationID',
        'payment_type',
        'fare_amount',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'improvement_surcharge',
        'total_amount',
        'congestion_surcharge', 
        'taxi_type'
    )

In [9]:
commons_cols = []

for col_green in df_green.columns:
    for col_yellow in df_yellow.columns:
        
        if col_green == col_yellow: 
            commons_cols.append(col_green)
        
        else: 
            None

In [10]:
commons_cols

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'taxi_type']

In [28]:
df_green = df_green.select(commons_cols)

In [29]:
df_yellow = df_yellow.select(commons_cols)

In [30]:
yellow_green_dataset_unioned = df_yellow.union(df_green)

In [35]:
df_trips_data_consolidated = \
    yellow_green_dataset_unioned \
    .withColumn('revenue_month', F.date_trunc('month', yellow_green_dataset_unioned.pickup_datetime)) \
    .select(
        #Grouping columns: 
        yellow_green_dataset_unioned.PULocationID.alias('pickup_zone_ID'), 
        F.col('revenue_month'), 

        #Aggregations columns: 
        yellow_green_dataset_unioned.total_amount, 
        yellow_green_dataset_unioned.PULocationID
    ) \
    .filter(F.col('revenue_month') >= '2019-01-01 00:00:00') \
    .groupBy(
        F.col('pickup_zone_ID'), 
        F.col('revenue_month')
    ) \
    .agg(
        F.sum(yellow_green_dataset_unioned.total_amount).alias('Total_revenue_monthly_yellow'), 
        F.count(yellow_green_dataset_unioned.PULocationID).alias('Total_trips_monthly_yellow')
    ) \
    .orderBy(
        F.desc(F.col('Total_revenue_monthly_yellow'))
    )

In [21]:
df_trips_data_consolidated \
    .repartition(20) \
    .write.parquet('/home/daniel/notebooks/taxi_data_reporting/revenue/trips_data_consolidated', 
                   mode= 'overwrite'
                  )